# Kaggle 07 — Full Evaluation + Ablation

**Week 7 task** — Run on Kaggle T4 (quantized model fits on T4 in AWQ 4-bit).

Evaluates all stages: vanilla → CPT → SFT → DPO → +retrieval

Benchmarks:
- COBOLEval (146 problems) — primary
- COBOL-JavaTrans (143 pairs) — translation
- MainframeBench held-out — supplementary


In [ ]:
!pip install uv -q
!uv pip install --system vllm>=0.21.0 datasets huggingface-hub gnucobol -q
!sudo apt-get install -y gnucobol -q


In [ ]:
import sys, os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

!git clone https://github.com/BloopAI/COBOLEval.git /kaggle/working/COBOLEval
!git clone https://github.com/YOUR_GH_USERNAME/Qwen-COBOL.git /kaggle/working/Qwen-COBOL
%cd /kaggle/working/Qwen-COBOL
sys.path.insert(0, '.')


In [ ]:
# Start vLLM with AWQ model (fits on T4 16GB in AWQ 4-bit ~8GB)
import subprocess, time
MODEL = 'YOUR_HF_USERNAME/qwen-cobol-27b-awq'
proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL, '--quantization', 'awq_marlin',
    '--max-model-len', '8192', '--port', '8000'
])
time.sleep(60)  # wait for server to start
print('vLLM server started')


In [ ]:
from src.eval.cobolceval_runner import run_eval
import argparse

args = argparse.Namespace(
    model=MODEL,
    coboleval_dir='/kaggle/working/COBOLEval',
    output='results/coboleval_final.json',
    thinking_budget=1024,
)
results = run_eval(args)
print(f"Compile rate: {results['compile_rate']*100:.1f}% | Pass@1: {results['pass_at_1']*100:.1f}%")


In [ ]:
from src.eval.leaderboard_report import generate_report
print(generate_report('results/coboleval_final.json'))
